# Dwarf Example 21: RMSE Improvement in Dwarfs

**EPS Research — Dwarf/Irregular HI Corpus v1.0**

Apply the omega correction to all 24 omega-ready dwarfs
and measure RMSE improvement vs Keplerian baseline.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20320362  
**Sources:** LVHIS (Koribalski 2019), VLA-ANGST (Ott 2012), LITTLE THINGS (Oh 2015), WALLABY DR2  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
with open('dwarf_irregular_corpus_v1.json') as f:
    corpus=json.load(f)
results=[]
for g in corpus['galaxies']:
    if not g.get('omega_ready') or not g.get('data') or len(g['data'])<3: continue
    d=g['data']; R=np.array([p['Rad'] for p in d]); V=np.array([p.get('Vrot', 0) for p in d])
    R1,V1=R[0],V[0]; R2,V2=R[-1],V[-1]
    if R1<=0 or R2<=0 or V1<=0 or V2<=0: continue
    omega=(V2/R2-V1/R1)*(R1/R2)**1.5
    V_adj=V-R*omega; V_kep=np.sqrt(V2**2*R2/R)
    rmse_o=np.sqrt(np.mean((V_adj-V)**2))
    rmse_k=np.sqrt(np.mean((V_kep-V)**2))
    results.append({'galaxy':g['galaxy'],'rmse_o':rmse_o,'rmse_k':rmse_k,
                    'improved':rmse_o<rmse_k,'omega':omega})
print(f"Dwarfs analyzed: {len(results)}")
print(f"Improved: {sum(1 for r in results if r['improved'])}/{len(results)}")
ro=[r['rmse_o'] for r in results]; rk=[r['rmse_k'] for r in results]
print(f"Mean RMSE omega:   {np.mean(ro):.1f} km/s")
print(f"Mean RMSE Kepler:  {np.mean(rk):.1f} km/s")
fig,ax=plt.subplots(figsize=(7,6))
ax.scatter(rk,ro,s=40,color='#2ca02c',alpha=0.8,zorder=3)
lim=[0,max(rk+ro)*1.05]
ax.plot(lim,lim,'k--',lw=1,alpha=0.4,label='1:1')
ax.set_xlabel('RMSE Keplerian (km/s)',fontsize=11); ax.set_ylabel('RMSE Omega (km/s)',fontsize=11)
ax.set_title('RMSE Improvement — Omega-Ready Dwarfs\nPoints below diagonal = improved',fontsize=10)
ax.legend(); plt.tight_layout()
plt.savefig('dw21_rmse_dwarfs.png',dpi=150,bbox_inches='tight'); plt.show()